# SunSim spec test - PHOENIX mu (new version)

Same run as `SunSim_PHOENIX_mu_test_sophie.ipynb` (old SunSim, `compute_forward(observables=['spec'])`), with the new SunSim:
TRAPPIST-1, 5 deg equatorial spot, `n_grid_rings` from the configuration file, Phoenix SPECINT models (`models/Phoenix_mu`),
5 epochs over one rotation. The same configuration file and spot map are read, and the same files are written.

What the old `spec` observable computes, and how it is done here:
* spectrum of one cell of every ring = Phoenix SPECINT intensity interpolated at the mu of the ring x projected area / 4 pi,
  below the smallest tabulated mu linearly to 0 at the limb: `flux_grid(..., mode='spectroscopy')`;
* no Doppler shift (`stellar_rotation=False`) and no filter (`convolve_spec_with_filt = False`);
* spot: Phoenix at `temperature_photosphere - spot_T_contrast`; faculae (if any): photosphere x bolometric limb brightening.

Differences with the old code (`REPRODUCE_OLD_BUGS = True` reproduces them; checked to 1e-13 on a test case):
1. Phoenix SPECINT models tabulate different mu angles (e.g. `lte02500-5.00` and `lte02600-5.00`). The old code added the rows of
   the 4 models of the T / logg interpolation without interpolating them in mu. Here every model is first interpolated in mu on the
   angles of the lowest-T, lowest-logg model.
2. The old `spec` branch reads the photosphere and then the spot model into the same `acd` variable, so the photosphere spectra are
   interpolated in mu with the angles of the spot model.
3. Not reproduced: the old spec kernel (`loop_generate_rotating_spec_nb`) does not use `active_region_types`, so faculae were
   treated as spots and its `ff_fc` is wrong. This benchmark has no faculae (`ff_fc = 0`), so it does not matter here.

With the default (`False`), the spectra differ from the old ones by ~0.1 % (test at 3100 K), almost the same at every epoch.

In [ ]:
import sys
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Change these to point to the new SunSim, the inputs of the old run and the Phoenix SPECINT models
SUNSIM_PATH = '/Users/sophiestucki/Documents/sunsim_new'
CONF = 'TRAPPIST1_spec_PHOENIX_mu.conf'                 # same file as the old run
SPOTMAP = 'spotmap.dat'                                # rows: type, t0, lifetime, colatitude, longitude, radius, ...
PHOENIX_MU = '/data/spotless/software/oporquer/src/sunsim_trappist_work/starsim/models/Phoenix_mu'
OLD_RESULTS = 'results_phoenix_mu'                        # output folder of the old run (for the comparison), or None
OUTPUT = 'results_phoenix_mu_new'
REPRODUCE_OLD_BUGS = False                        # True: mu handling of the old code (see above)

sys.path.insert(0, SUNSIM_PATH)
import sunsim
from sunsim import StarSim, flux_grid, read_old_config, limb_brightening_factor
from sunsim.old_config import _phoenix_specint, _phoenix_specint_old, _regions
print('Using:', sunsim.__file__)

In [ ]:
cfg, report = read_old_config(CONF, observables=['lc'], paths={'phoenix_mu_folder': PHOENIX_MU})
spots = np.loadtxt(SPOTMAP, ndmin=2)
times = np.linspace(0, cfg['rotation_period'], 5, endpoint=False)
print('Times [days]:', times)
print('T_ph = %g K, T_sp = %g K, logg = %g, N_rings = %d, %g-%g A' % (
    cfg['temperature_photosphere'], cfg['temperature_photosphere'] - cfg['spot_T_contrast'], cfg['logg'],
    cfg['n_grid_rings'], cfg['wavelength_lower_limit'], cfg['wavelength_upper_limit']))

In [ ]:
def run_spec(cfg, spots, times, reproduce_old_bugs=False):
    """Disc-integrated spectrum at every epoch, as the old compute_forward(observables=['spec'])."""
    N = cfg['n_grid_rings']
    lo, hi = cfg['wavelength_lower_limit'], cfg['wavelength_upper_limit']
    T_ph = cfg['temperature_photosphere']
    T_sp = T_ph - cfg['spot_T_contrast']
    regions = _regions(cfg, spots)

    # Phoenix SPECINT intensities {mu: {'wav', 'intensity'}} interpolated in T and logg
    specint = _phoenix_specint_old if reproduce_old_bugs else _phoenix_specint
    acd_sp = specint(cfg, T_sp, lo, hi)[0]
    def phoenix(T):
        mu, wav, rows = specint(cfg, T, lo, hi)
        if reproduce_old_bugs:
            mu = acd_sp              # old: the angles of the spot model are used for every surface
        return {float(m): {'wav': wav, 'intensity': r} for m, r in zip(mu, rows)}

    def grid(T):
        g = flux_grid(N, phoenix(T), 'x', None, lo, hi, mode='spectroscopy', stellar_rotation=False)
        g.built_grid()
        return g

    g_ph, g_sp = grid(T_ph), grid(T_sp)
    # faculae: photosphere x bolometric limb brightening at the mu of every cell
    if any(r.type == 'fc' for r in regions):
        law = 'meunier' if cfg['meunier'] else 'starsim'
        mu_cells = np.repeat(g_ph.amu, g_ph.Ngrid_in_ring)
        grid_fc = g_ph.grid * limb_brightening_factor(mu_cells, T_ph, law)[:, None]
    else:
        grid_fc = g_ph.grid

    B = cfg['differential_rotation']
    ss = StarSim(times, N, inclination=np.deg2rad(90 - cfg['inclination']), rotation_period=cfg['rotation_period'],
                 differential_rotation=(1.698 * B / 2.66, 2.346 * B / 2.66), stellar_radius=cfg['radius'],
                 active_regions_map=regions, mode=['spectroscopy'],
                 flux_grid_qp=g_ph.grid, flux_grid_sp=g_sp.grid, flux_grid_fc=grid_fc)
    ss.generate_timeseries(verbose=False)
    return {'spec_wv': g_ph.wv, 'spec': ss.spec_var, 'ff_ph': ss.ff_quiet, 'ff_sp': ss.ff_sp,
            'ff_fc': ss.ff_fc, 'ff_pl': ss.ff_pl, 'starsim': ss}

In [ ]:
# The first call includes compilation time (and the reading of the Phoenix models, kept in memory), so repeat.
# As in the old run, the time includes the preparation of the spectra and grids, not only the time series.
elapsed = []
for label in ['First call', 'Repeat']:
    start = perf_counter()
    res = run_spec(cfg, spots, times, REPRODUCE_OLD_BUGS)
    elapsed.append(perf_counter() - start)
    print(f'{label}: {elapsed[-1]:.3f} s')
print('time series only (Numba loop):', '%.3f s' % res['starsim'].timing['timeseries'])
for t, p, s, f in zip(times, res['ff_ph'], res['ff_sp'], res['ff_fc']):
    print(f'Date {t:.2f}. ff_ph={p:.3f}%. ff_sp={s:.3f}%. ff_fc={f:.3f}%.')

In [ ]:
output = Path(OUTPUT)
output.mkdir(exist_ok=True)
wavelength = res['spec_wv']     # Angstrom
spectra = res['spec']           # Shape: (time, wavelength)

pd.DataFrame(spectra.T, index=wavelength,
             columns=[f'flux_t{t:.6f}_day' for t in times]).rename_axis(
                 'wavelength_A').to_csv(output / 'spectra.csv')
pd.DataFrame({'time_day': times, **{f'{k}_percent': res[k]
             for k in ['ff_ph', 'ff_sp', 'ff_fc', 'ff_pl']}}).to_csv(
                 output / 'filling_factors.csv', index=False)
pd.DataFrame({'call': ['first', 'repeat'], 'seconds': elapsed}).to_csv(
    output / 'timings.csv', index=False)
np.savez_compressed(output / 'spectra.npz', time=times,
                    spec_wv=wavelength, spec=spectra)
print('Saved to:', output.resolve())

In [ ]:
for t, spectrum in zip(times, spectra):
    plt.plot(wavelength / 1e4, spectrum, label=f'{t:.2f} d', alpha=0.5)
plt.xlabel('Wavelength [µm]')
plt.ylabel('Flux')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
for t, spectrum in zip(times, spectra):
    plt.plot(wavelength / 1e4, spectrum-np.mean(spectra,axis=0), label=f'{t:.2f} d')
plt.xlabel('Wavelength [µm]')
plt.ylabel('Delta flux')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fluxes = np.sum(spectra,axis=1)
fluxes /= np.median(fluxes)
plt.scatter(times, fluxes)
plt.xlabel('Time [day]')
plt.ylabel('Norm. white light flux')
plt.tight_layout()
plt.show()

## Comparison with the old run

Reads the output folder of the old notebook (`OLD_RESULTS`: `spectra.npz`, `filling_factors.csv`, `timings.csv`).

In [ ]:
old_dir = Path(OLD_RESULTS) if OLD_RESULTS else None
if old_dir is None or not (old_dir / 'spectra.npz').exists():
    print('No old results in', old_dir, ': comparison skipped')
else:
    old = np.load(old_dir / 'spectra.npz')
    old_ff = pd.read_csv(old_dir / 'filling_factors.csv')
    old_time = pd.read_csv(old_dir / 'timings.csv')
    assert np.allclose(old['time'], times), 'not the same epochs'
    assert np.array_equal(old['spec_wv'], wavelength), 'not the same wavelengths'
    rel = spectra / old['spec'] - 1

    print('Timings [s]:')
    print(pd.DataFrame({'call': ['first', 'repeat'], 'old': old_time['seconds'].values, 'new': elapsed,
                        'speed-up': old_time['seconds'].values / np.array(elapsed)}).to_string(index=False))
    print('\nFilling factors [%] (new - old):')
    print(pd.DataFrame({'time_day': times, **{k: res[k] - old_ff[f'{k}_percent'].values
                                              for k in ['ff_ph', 'ff_sp', 'ff_fc', 'ff_pl']}}).to_string(index=False))
    print('\nSpectra, new / old - 1: max |.| = %.2e, median |.| = %.2e' % (np.nanmax(np.abs(rel)), np.nanmedian(np.abs(rel))))
    wl_old = old['spec'].sum(axis=1)
    print('White light, new / old - 1:', spectra.sum(axis=1) / wl_old - 1)

    fig, axs = plt.subplots(3, 1, figsize=(8, 9), sharex=True)
    for t, s_new, s_old, r in zip(times, spectra, old['spec'], rel):
        axs[0].plot(wavelength / 1e4, s_new, label=f'{t:.2f} d', alpha=0.5)
        axs[1].plot(wavelength / 1e4, (s_new - spectra.mean(axis=0)) - (s_old - old['spec'].mean(axis=0)), lw=0.5)
        axs[2].plot(wavelength / 1e4, r, lw=0.5)
    axs[0].set_ylabel('Flux (new)'); axs[0].legend()
    axs[1].set_ylabel('Delta flux: new - old')
    axs[2].set_ylabel('new / old - 1')
    axs[2].set_xlabel('Wavelength [µm]')
    plt.tight_layout()
    plt.show()

    plt.scatter(times, wl_old / np.median(wl_old), label='old')
    plt.scatter(times, fluxes, marker='x', label='new')
    plt.xlabel('Time [day]')
    plt.ylabel('Norm. white light flux')
    plt.legend()
    plt.tight_layout()
    plt.show()